# TV Ad Attribution & Media Mix Model

**Full Pipeline Notebook** — Run in Google Colab

This notebook generates realistic TV advertising data for a DTC brand, builds multiple attribution models (LightGBM counterfactual, Markov chain, heuristic), trains both frequentist and Bayesian Media Mix Models, performs SHAP explainability analysis, cross-channel effect modeling, and a stacked comparison of all 6 approaches.

## Brand Profile
- Consumer DTC brand (mattress/meal-kit category)
- $15M annual ad budget across TV + digital
- 8 DMAs: New York, Los Angeles, Chicago, Houston, Phoenix, Philadelphia, Dallas, Atlanta
- Campaign: January–December 2023
- TV: ABC, CBS, NBC, FOX + ESPN, CNN, HGTV, Food Network, TNT, TBS
- Digital: Paid Search, Social (Meta/YouTube), Display

## ML Techniques
LightGBM, Ridge Regression, Bayesian MCMC (PyMC), TreeSHAP, Absorbing Markov Chains, Granger Causality, Mediation Analysis, Hill Saturation, Geometric Adstock, SLSQP Optimization

## Setup

In [ ]:
!pip install -q numpy pandas scikit-learn lightgbm joblib matplotlib scipy pymc arviz shap statsmodels

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

np.random.seed(42)
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

print('Setup complete')

---
## Part 1: Data Generation

Generate ~21K ad airing records, ~840K web traffic rows, and 52 weeks of channel spend data.

In [ ]:
# Run the data generator
# (If running in Colab, upload src/generate_tv_data.py first,
# or copy the code directly)

import sys
sys.path.insert(0, '.')

from src.generate_tv_data import main as generate_data
airings, traffic, weekly_spend = generate_data()

In [ ]:
# Quick data exploration
print(f'Airings shape: {airings.shape}')
print(f'Traffic shape: {traffic.shape}')
print(f'Weekly spend shape: {weekly_spend.shape}')
print(f'\nTotal TV cost: ${airings["cost"].sum():,.0f}')
print(f'Total sessions: {traffic["sessions"].sum():,}')
print(f'Total annual spend: ${weekly_spend["total_spend"].sum():,.0f}')
print(f'Total annual revenue: ${weekly_spend["total_revenue"].sum():,.0f}')

airings.head()

In [ ]:
# Airing distribution by network and daypart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

airings['network'].value_counts().plot.barh(ax=axes[0], color='#3498db')
axes[0].set_title('Airings by Network')
axes[0].set_xlabel('Count')

airings['daypart'].value_counts().plot.barh(ax=axes[1], color='#2ecc71')
axes[1].set_title('Airings by Daypart')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Weekly spend over time
spend_cols = ['tv_broadcast_spend', 'tv_cable_spend', 'tv_streaming_spend',
              'paid_search_spend', 'social_spend', 'display_spend']

fig, ax = plt.subplots(figsize=(12, 5))
weekly_spend[spend_cols].div(1e3).plot.area(ax=ax, alpha=0.7)
ax.set_xlabel('Week')
ax.set_ylabel('Spend ($K)')
ax.set_title('Weekly Channel Spend')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

print(f'Revenue/Spend ROAS: {weekly_spend["total_revenue"].sum() / weekly_spend["total_spend"].sum():.2f}x')

---
## Part 2: Attribution Model

### Step 1: Baseline Model (Counterfactual)
Train a LightGBM model to predict web traffic in the absence of TV ads.
Train on "clean" periods (no TV airing in prior 30 minutes).

In [ ]:
from src.attribution import main as run_attribution
attribution_df = run_attribution()

In [ ]:
# Attribution results
attribution_df = pd.read_csv('data/airing_attribution.csv')
print(f'Total incremental sessions: {attribution_df["incremental_sessions"].sum():,.0f}')
print(f'Total incremental revenue: ${attribution_df["incremental_revenue"].sum():,.0f}')
print(f'Average ROAS (positive): {attribution_df[attribution_df["roas"] > 0]["roas"].mean():.2f}x')

# Show baseline vs actual plot
from IPython.display import Image
Image('results/baseline_vs_actual.png')

In [ ]:
# ROAS by network
Image('results/roas_by_network.png')

In [ ]:
# ROAS by daypart
Image('results/roas_by_daypart.png')

---
## Part 3: Media Mix Model

### Adstock Transformation
Model carryover effects: TV ads today still influence behavior next week.

### Hill Saturation
Model diminishing returns: the 10th million in TV has less impact than the 1st.

### Ridge Regression
Stable regression that handles collinearity between correlated channel spends.

In [ ]:
from src.media_mix import main as run_mmm
mmm_model, mmm_scaler, contributions_df, optimal_df = run_mmm()

In [ ]:
# Channel contributions
contributions_df = pd.read_csv('data/channel_contributions.csv')
contributions_df.sort_values('contribution', ascending=False)

In [ ]:
# Actual vs predicted revenue
Image('results/actual_vs_predicted.png')

In [ ]:
# Channel contributions chart
Image('results/channel_contributions.png')

In [ ]:
# Saturation curves
Image('results/saturation_curves.png')

In [ ]:
# Adstock decay curves
Image('results/adstock_curves.png')

In [ ]:
# Budget optimization results
optimal_df = pd.read_csv('data/optimal_allocation.csv')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(optimal_df['budget'] / 1e6, optimal_df['current_mix_revenue'] / 1e6,
        'o--', color='#e74c3c', label='Current Mix')
ax.plot(optimal_df['budget'] / 1e6, optimal_df['optimized_revenue'] / 1e6,
        's-', color='#2ecc71', label='Optimized Mix')
ax.fill_between(optimal_df['budget'] / 1e6,
                optimal_df['current_mix_revenue'] / 1e6,
                optimal_df['optimized_revenue'] / 1e6,
                alpha=0.1, color='#2ecc71')
ax.set_xlabel('Total Budget ($M)')
ax.set_ylabel('Predicted Revenue ($M)')
ax.set_title('Budget Optimization: Current vs Optimal Allocation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nLift from optimization by budget level:')
for _, row in optimal_df.iterrows():
    print(f"  ${row['budget']/1e6:.0f}M budget: {row['lift_pct']:.1f}% lift")

---
## Part 4: Advanced Analysis

### 4A. SHAP Explainability
TreeSHAP provides exact Shapley values for the LightGBM baseline model — theoretically grounded, additive, and locally accurate feature importance.

In [ ]:
from src.shap_analysis import main as run_shap
run_shap()

In [ ]:
# SHAP feature importance
shap_imp = pd.read_csv('data/shap_feature_importance.csv')
print('Top 10 features by SHAP importance:')
shap_imp.head(10)

# Show SHAP summary plot
Image('results/shap_summary_baseline.png')

### 4B. Markov Chain Attribution
Absorbing Markov chains model the customer journey as a stochastic process. Removal effects measure each channel's importance by simulating what happens when it's removed entirely.

In [ ]:
from src.markov_attribution import main as run_markov
run_markov()

In [ ]:
# Markov vs Last-Touch vs First-Touch attribution
markov_df = pd.read_csv('data/markov_attribution.csv')
markov_df

# Show comparison plot
Image('results/markov_vs_lasttouch.png')

### 4C. Cross-Channel Effects (TV → Search)
Investigating how TV advertising drives online search behavior using Granger causality, mediation analysis, and interaction modeling.

In [ ]:
from src.cross_channel import main as run_cross_channel
run_cross_channel()

In [ ]:
# Cross-channel results
import json
with open('data/cross_channel_effects.json') as f:
    effects = json.load(f)

print(f"Peak TV→Search correlation: r={effects['peak_correlation']['correlation']:.3f} at lag {effects['peak_correlation']['lag_weeks']} weeks")
print(f"Mediation: {effects['mediation']['indirect_effect'] / effects['mediation']['total_effect'] * 100:.0f}% of TV effect mediated through search")
print(f"TV×Search interaction coefficient: ${effects['interaction']['interaction_coefficient']:,.0f}")

Image('results/tv_search_lag_correlation.png')

### 4D. Bayesian MMM (PyMC)
Full Bayesian estimation via MCMC sampling. Unlike the frequentist approach, this provides posterior distributions over ROAS — capturing uncertainty with credible intervals, not just point estimates.

In [ ]:
from src.bayesian_mmm import main as run_bayesian
run_bayesian()

In [ ]:
# Bayesian ROAS posteriors with credible intervals
bayes_roas = pd.read_csv('data/bayesian_roas_posteriors.csv')
print('Posterior ROAS (90% Credible Intervals):')
for _, row in bayes_roas.iterrows():
    print(f"  {row['channel']}: {row['median_roas']:.2f}x [{row['ci_5']:.2f}, {row['ci_95']:.2f}]")

Image('results/bayesian_posterior_roas.png')

### 4E. Stacked Model Comparison
Six attribution methods applied to the same data — revealing where models agree (higher confidence) and where they disagree (more uncertainty).

In [ ]:
from src.model_comparison import main as run_comparison
run_comparison()

---
## Summary

### Key Findings

1. **Attribution**: The LightGBM baseline model (R² ≈ 0.67) identifies TV-driven traffic lift. ~188K incremental sessions attributed to TV.

2. **Frequentist MMM**: Ridge regression (R² ≈ 0.90) decomposes revenue into channel effects with adstock and Hill saturation curves.

3. **Bayesian MMM**: PyMC MCMC provides posterior ROAS distributions with 90% credible intervals — uncertainty quantification, not just point estimates.

4. **SHAP Explainability**: TreeSHAP provides exact Shapley values showing DMA and time-of-day as dominant traffic drivers.

5. **Markov Attribution**: Absorbing Markov chains capture TV's role as a journey initiator (20% attribution vs 5% last-touch).

6. **Cross-Channel Effects**: TV spend drives search behavior (r=0.88), with almost all TV effect mediated through search.

7. **Model Comparison**: Six models produce different attributions — TV value varies 2-5x by method, underscoring why methodology matters.

8. **Budget Optimization**: SLSQP optimizer suggests 10-28% revenue lift through channel reallocation.

### ML Techniques Used
- **LightGBM** for counterfactual baseline prediction
- **Geometric adstock** for carryover effect modeling
- **Hill saturation curves** for diminishing returns
- **Ridge regression** for stable multi-channel attribution
- **Bayesian inference (PyMC/MCMC)** for posterior ROAS with credible intervals
- **TreeSHAP** for exact feature-level Shapley value explanations
- **Absorbing Markov chains** with removal effects for journey-based attribution
- **Granger causality** for temporal causal inference
- **Mediation analysis** for direct/indirect effect decomposition
- **SLSQP constrained optimization** for budget allocation
- **Time-series cross-validation** for model evaluation

---
## Summary

### Key Findings

1. **Attribution**: The baseline model (R² ≈ 0.67) successfully identifies TV-driven traffic lift. ~188K incremental sessions attributed to TV.

2. **Channel Contributions**: The MMM (R² ≈ 0.90) decomposes revenue into channel effects. Paid Search shows highest marginal ROI, while TV drives longer-term brand effects captured through adstock.

3. **Budget Optimization**: The optimizer suggests reallocating budget across channels for 10-28% revenue lift depending on budget level.

### ML Techniques Used
- **LightGBM** for counterfactual baseline prediction
- **Geometric adstock** for carryover effect modeling
- **Hill saturation curves** for diminishing returns
- **Ridge regression** for stable multi-channel attribution
- **SLSQP constrained optimization** for budget allocation
- **Time-series cross-validation** for model evaluation